# TP POO — Simulation d'un écosystème proie-prédateur

## Rapport de conception et tests unitaires

Ce notebook s'appuie uniquement sur l'organisation et les mécanismes présents dans le code Python fourni.

## 1. Organisation générale du programme

Le programme modélise trois types d'objets principaux : `plante`, `herbivore` et `carnivore`. La classe `animale` regroupe les caractéristiques et comportements communs aux animaux. `herbivore` et `carnivore` héritent de cette classe et redéfinissent notamment leur méthode `action()`. La simulation est ensuite gérée par les fonctions `Simulation()`, `run_simulation()` et `repousse_plantes()`, tandis que `lancer_dashboard()` assure la visualisation. (Visualisation générée par IA)

## 2. Héritage

L'héritage est utilisé entre `animale` et ses deux classes filles `herbivore` et `carnivore`.

`animale` contient les éléments communs : position, espèce, vitesse, vision, réserve de nourriture, âge, état de vie, déplacement, alimentation, reproduction et mort.

Les classes `herbivore` et `carnivore` récupèrent ces fonctionnalités avec `super().__init__()` et adaptent leurs paramètres biologiques (`AGE_MATURE`, `ESPERANCE`, `REPRO_PAUSE`, etc.) ainsi que leur méthode `action()`.

**Choix justifié :** un herbivore et un carnivore sont tous les deux des animaux dans la logique du programme, donc factoriser leurs comportements communs dans `animale` évite de dupliquer le code.

## 3. Polymorphisme

Le polymorphisme apparaît notamment avec `action()`. La fonction `Simulation()` parcourt séparément les listes d'herbivores et de carnivores et appelle `herbivore.action(plante_list)` ou `carnivore.action(herbivore_list)`.

Les deux méthodes portent donc le même nom mais leur comportement dépend de la classe de l'objet : l'herbivore recherche des plantes alors que le carnivore recherche des herbivores.

Le polymorphisme apparaît également dans `manger()` : cette méthode est définie une seule fois dans `animale` mais utilise les paramètres de classe (`PART_CORPS`, `SUCCES_CHASSE`) propres au type d'animal.

## 4. Encapsulation

L'encapsulation est **partielle** dans le code actuel. Les données sont regroupées dans les objets (`pos`, `age`, `nourriture`, `vivant`, etc.) et les modifications passent souvent par des méthodes comme `mort()`, `manger()`, `deplacement()` ou `repro()`.

Cependant, les attributs ne sont pas privés : ils restent directement accessibles depuis l'extérieur. Par exemple, `self.nourriture` ou `self.vivant` sont modifiés directement dans les méthodes. Le code ne met donc pas en place une encapsulation stricte avec des attributs privés/propriétés.

## 5. Composition

La composition est présente dans la gestion des populations : les listes `plante_list`, `herbivore_list` et `carnivore_list` contiennent les objets qui constituent l'écosystème simulé.

La fonction `run_simulation()` crée ces collections avec `initialisation()` puis les fait évoluer pendant les jours de simulation. La simulation est donc construite à partir de collections d'objets plantes et animaux.

Il faut toutefois noter qu'il n'existe pas dans le code une classe `Environnement` contenant explicitement ces objets : la composition est donc réalisée ici par les listes et les fonctions de simulation plutôt que par une classe dédiée.

## 6. Répartition des responsabilités

- **`plante`** : représente une plante, sa position, sa quantité de nourriture et son état de vie.
- **`animale`** : contient les comportements communs aux animaux : déplacement, alimentation, reproduction et mort, ainsi que les paramètres généraux.
- **`herbivore`** : définit les paramètres propres aux herbivores et leur cycle d'action.
- **`carnivore`** : définit les paramètres propres aux carnivores et leur cycle d'action.
- **`initialisation()`** : crée les populations initiales.
- **`Simulation()`** : exécute un tour de simulation et ajoute les nouveaux individus.
- **`dep_age()`** et **`proportion_reserve_proie()`** : réalisent des calculs indépendants des objets.
- **`repousse_plantes()`** : gère la régénération des plantes.
- **`run_simulation()`** : enchaîne les tours et enregistre l'historique.
- **`lancer_dashboard()`** : affiche et contrôle graphiquement les résultats de la simulation.

Cette séparation permet notamment de distinguer la logique des animaux, la logique globale de simulation et la visualisation.

## 7. Tests unitaires

Les tests ci-dessous vérifient des comportements directement présents dans le code : création d'un herbivore, mort, déplacement, alimentation, reproduction et exécution d'un tour de simulation.

In [2]:
import importlib.util
import random
import unittest

spec = importlib.util.spec_from_file_location('simulation', 'POO_simulation_vie.py')
simulation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(simulation)


In [22]:
class TestSimulation(unittest.TestCase):

    def test_creation_herbivore(self):
        h = simulation.herbivore((10, 10), 'lapin', 3, 9, 20)
        self.assertEqual(h.age, 1)
        self.assertEqual(h.nourriture, 20 * h.NOURRITURE_INIT)
        self.assertTrue(h.vivant)

    def test_mort(self):
        h = simulation.herbivore((10, 10), 'lapin', 3, 9, 20)
        h.mort()
        self.assertFalse(h.vivant)

    def test_deplacement_consomme_nourriture(self):
        h = simulation.herbivore((10, 10), 'lapin', 3, 9, 20)
        nourriture_initiale = h.nourriture
        h.deplacement([])
        self.assertNotEqual(h.pos, (10, 10))
        self.assertLess(h.nourriture, nourriture_initiale)

    def test_manger(self):               
        c = simulation.carnivore((10, 10), 'renard', 3, 12, 30)
        h = simulation.herbivore((10, 10), 'lapin', 3, 9, 20)

        nourriture_initiale = c.nourriture

        c.SUCCES_CHASSE = 1.0   # chasse toujours réussie, sinon rend faut certaines fois

        c.manger(h)

        self.assertFalse(h.vivant)
        self.assertGreater(c.nourriture, nourriture_initiale)

    def test_reproduction(self):
        h = simulation.herbivore((10, 10), 'lapin', 3, 9, 20, nourriture=20)
        h.age = h.AGE_MATURE
        h.trepro = h.REPRO_PAUSE + 1

        enfant = h.repro()

        self.assertIsNotNone(enfant)
        self.assertEqual(enfant.espece, h.espece)
        self.assertEqual(enfant.age, 1)

    def test_simulation(self):
        plantes, herbivores, carnivores = simulation.initialisation(20, 5, 2)
        simulation.Simulation(plantes, herbivores, carnivores)

        self.assertIsInstance(plantes, list)
        self.assertIsInstance(herbivores, list)
        self.assertIsInstance(carnivores, list)
        self.assertTrue(all(h.vivant for h in herbivores))
        self.assertTrue(all(c.vivant for c in carnivores))


unittest.TextTestRunner(verbosity=2).run(unittest.defaultTestLoader.loadTestsFromTestCase(TestSimulation))

test_creation_herbivore (__main__.TestSimulation.test_creation_herbivore) ... ok
test_deplacement_consomme_nourriture (__main__.TestSimulation.test_deplacement_consomme_nourriture) ... ok
test_manger (__main__.TestSimulation.test_manger) ... ok
test_mort (__main__.TestSimulation.test_mort) ... ok
test_reproduction (__main__.TestSimulation.test_reproduction) ... ok
test_simulation (__main__.TestSimulation.test_simulation) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.003s

OK


<unittest.runner.TextTestResult run=6 errors=0 failures=0>